In [ ]:
# ============================================================
# BLOQUE 03 - AJUSTE DE DISTRIBUCIONES MARGINALES
# Momentos-L como inicialización + MLE final + AIC/BIC
# Duración y severidad de eventos de sequía nSPEI-3
# ============================================================

if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  tidyverse,  # Manipulación de datos.
  janitor,    # Estandarización de nombres.
  lmomco      # Momentos-L y distribuciones probabilísticas.
)


In [ ]:
# ============================================================
# RUTAS DEL PROYECTO
# ============================================================

encontrar_raiz_proyecto <- function(inicio = getwd()) {
  marcadores <- c("01_datos", "02_scripts", "03_resultados", "04_documentacion")
  es_raiz <- function(x) all(dir.exists(file.path(x, marcadores)))

  raiz_env <- Sys.getenv("TESIS_SEQUIAS_ROOT", unset = "")
  if (nzchar(raiz_env)) {
    raiz_env <- normalizePath(raiz_env, winslash = "/", mustWork = FALSE)
    if (es_raiz(raiz_env)) return(raiz_env)
  }

  ruta <- normalizePath(inicio, winslash = "/", mustWork = TRUE)

  repeat {
    if (es_raiz(ruta)) return(ruta)

    padre <- dirname(ruta)
    if (identical(padre, ruta)) {
      stop(
        "No se encontró la raíz del proyecto. ",
        "Ejecute Jupyter dentro del repositorio o defina TESIS_SEQUIAS_ROOT."
      )
    }

    ruta <- padre
  }
}

ruta_base <- encontrar_raiz_proyecto()

ruta_procesados <- file.path(
  ruta_base, "01_datos", "procesados"
)

ruta_tablas <- file.path(
  ruta_base, "03_resultados", "03_marginales", "tablas"
)

archivo_eventos <- file.path(
  ruta_procesados,
  "eventos_nspei_todas_escalas.csv"
)

dir.create(ruta_tablas, recursive = TRUE, showWarnings = FALSE)

cat("Archivo de eventos:", file.exists(archivo_eventos), "\n")


In [ ]:
# ============================================================
# EVENTOS DE SEQUÍA nSPEI-3
# ============================================================

eventos <- read.csv(
  archivo_eventos,
  stringsAsFactors = FALSE
) %>%
  clean_names()

eventos_3 <- eventos %>%
  filter(escala == "nspei_3")

c1 <- eventos_3 %>% filter(cluster_ward == 1)
c2 <- eventos_3 %>% filter(cluster_ward == 2)
c3 <- eventos_3 %>% filter(cluster_ward == 3)

conteo_eventos <- eventos_3 %>%
  count(cluster_ward, name = "n_eventos")

conteo_eventos
cat("Total de eventos nSPEI-3:", nrow(eventos_3), "\n")


In [ ]:
# ============================================================
# AJUSTE MARGINAL
# Momentos-L como inicialización + MLE final + AIC/BIC
# Versión 2: tratamiento explícito de Weibull de 3 parámetros
# ============================================================

# NOTA METODOLÓGICA:
# 1) Los momentos-L se usan únicamente para obtener valores iniciales.
# 2) Los parámetros finales se estiman por máxima verosimilitud (MLE).
# 3) AIC y BIC se calculan con la log-verosimilitud maximizada.
# 4) No se eliminan observaciones con densidad inválida.
# 5) Para WEI se usa mle2par() de lmomco y se verifica explícitamente
#    convergencia, soporte y finitud de la densidad.

pdf_lmomco <- function(x, distribucion, par) {
  switch(
    distribucion,
    "wei" = pdfwei(x, par),
    "pe3" = pdfpe3(x, par),
    "ln3" = pdfln3(x, par),
    "gam" = pdfgam(x, par),
    "gpa" = pdfgpa(x, par),
    stop("Distribución no implementada: ", distribucion)
  )
}

# Convierte parámetros naturales a una escala sin restricciones.
a_escala_z <- function(par_nat, distribucion, x) {

  min_x <- min(x)
  s_x <- sd(x)
  if (!is.finite(s_x) || s_x <= 0) s_x <- 1

  eps <- max(1e-8, 1e-8 * max(1, abs(min_x), s_x))

  switch(
    distribucion,

    # En lmomco la Weibull usa la convención equivalente a
    # pweibull(x + zeta, shape = delta, scale = beta);
    # por ello el soporte exige x + zeta > 0.
    "wei" = {
      zeta <- par_nat[1]
      beta <- max(par_nat[2], eps)
      delta <- max(par_nat[3], eps)

      gap <- min_x + zeta
      if (!is.finite(gap) || gap <= eps) {
        gap <- max(0.05 * s_x, 0.01)
      }

      c(log(gap), log(beta), log(delta))
    },

    "ln3" = {
      zeta <- par_nat[1]
      mu <- par_nat[2]
      sigma <- max(par_nat[3], eps)

      gap <- min_x - zeta
      if (!is.finite(gap) || gap <= eps) {
        gap <- max(0.05 * s_x, 0.01)
      }

      c(log(gap), mu, log(sigma))
    },

    "pe3" = {
      mu <- par_nat[1]
      sigma <- max(par_nat[2], eps)
      gamma <- par_nat[3]

      c(mu, log(sigma), gamma)
    },

    "gam" = {
      alpha <- max(par_nat[1], eps)
      beta <- max(par_nat[2], eps)

      c(log(alpha), log(beta))
    },

    "gpa" = {
      xi <- par_nat[1]
      alpha <- max(par_nat[2], eps)
      k <- par_nat[3]

      gap <- min_x - xi
      if (!is.finite(gap) || gap <= eps) {
        gap <- max(0.05 * s_x, 0.01)
      }

      c(log(gap), log(alpha), k)
    },

    stop("Distribución no implementada: ", distribucion)
  )
}

desde_escala_z <- function(z, distribucion, x) {

  min_x <- min(x)

  switch(
    distribucion,

    "wei" = c(
      -min_x + exp(z[1]), # zeta: garantiza min(x) + zeta > 0
      exp(z[2]),          # beta
      exp(z[3])           # delta
    ),

    "ln3" = c(
      min_x - exp(z[1]),  # zeta
      z[2],               # mu_log
      exp(z[3])           # sigma_log
    ),

    "pe3" = c(
      z[1],               # mu
      exp(z[2]),          # sigma
      z[3]                # gamma
    ),

    "gam" = c(
      exp(z[1]),          # alpha
      exp(z[2])           # beta
    ),

    "gpa" = c(
      min_x - exp(z[1]),  # xi
      exp(z[2]),          # alpha
      z[3]                # kappa
    ),

    stop("Distribución no implementada: ", distribucion)
  )
}

nll_marginal <- function(z, x, distribucion) {

  par_nat <- desde_escala_z(z, distribucion, x)

  par_obj <- tryCatch(
    lmomco::vec2par(par_nat, type = distribucion),
    error = function(e) NULL
  )

  if (is.null(par_obj)) return(1e100)

  densidad <- tryCatch(
    pdf_lmomco(x, distribucion, par_obj),
    error = function(e) rep(NA_real_, length(x))
  )

  if (
    length(densidad) != length(x) ||
    any(!is.finite(densidad)) ||
    any(densidad <= 0)
  ) {
    return(1e100)
  }

  valor <- -sum(log(densidad))

  if (!is.finite(valor)) return(1e100)

  valor
}

generar_multistart <- function(z0) {

  p <- length(z0)

  desplazamientos <- list(
    rep(0, p),
    rep(0.10, p),
    rep(-0.10, p),
    seq(-0.20, 0.20, length.out = p),
    seq(0.20, -0.20, length.out = p)
  )

  lapply(desplazamientos, function(d) z0 + d)
}

# Ajuste especial para Weibull de 3 parámetros.
# Se considera válido solo si:
# - optim reporta convergencia 0,
# - todas las densidades son finitas y > 0,
# - el soporte contiene estrictamente toda la muestra.
ajustar_wei_mle <- function(x, par_lmom) {

  n <- length(x)

  fit <- tryCatch(
    lmomco::mle2par(
      x,
      type = "wei",
      init.para = par_lmom,
      silent = TRUE,
      null.on.not.converge = FALSE,
      method = "Nelder-Mead",
      control = list(
        maxit = 50000,
        reltol = 1e-10
      )
    ),
    error = function(e) NULL
  )

  if (is.null(fit)) {
    return(
      tibble(
        modelo = "WEI",
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = NA_integer_,
        Metodo = "lmomco::mle2par",
        Estado = "No válido: error de optimización",
        Margen_soporte = NA_real_,
        Parametros = NA_character_,
        Inicio_Lmom = paste(round(par_lmom$para, 6), collapse = ", ")
      )
    )
  }

  codigo <- tryCatch(fit$optim$convergence, error = function(e) NA_integer_)

  densidad <- tryCatch(
    lmomco::pdfwei(x, fit),
    error = function(e) rep(NA_real_, n)
  )

  # En la parametrización de lmomco el soporte requiere x + zeta > 0.
  margen_soporte <- min(x + fit$para[1])

  densidad_ok <- (
    length(densidad) == n &&
    all(is.finite(densidad)) &&
    all(densidad > 0)
  )

  soporte_ok <- is.finite(margen_soporte) && margen_soporte > 1e-8
  convergio_ok <- is.finite(codigo) && codigo == 0

  valido <- convergio_ok && densidad_ok && soporte_ok

  if (!valido) {

    motivo <- paste(
      c(
        if (!convergio_ok) "sin convergencia",
        if (!densidad_ok) "densidad no válida",
        if (!soporte_ok) "solución en/fuera del límite del soporte"
      ),
      collapse = "; "
    )

    return(
      tibble(
        modelo = "WEI",
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = codigo,
        Metodo = "lmomco::mle2par",
        Estado = paste0("No válido: ", motivo),
        Margen_soporte = margen_soporte,
        Parametros = paste(round(fit$para, 6), collapse = ", "),
        Inicio_Lmom = paste(round(par_lmom$para, 6), collapse = ", ")
      )
    )
  }

  loglik <- sum(log(densidad))
  k <- length(fit$para)

  tibble(
    modelo = "WEI",
    n = n,
    logLik = loglik,
    AIC = -2 * loglik + 2 * k,
    BIC = -2 * loglik + log(n) * k,
    Convergio = TRUE,
    Codigo_convergencia = codigo,
    Metodo = "lmomco::mle2par",
    Estado = "Válido",
    Margen_soporte = margen_soporte,
    Parametros = paste(round(fit$para, 6), collapse = ", "),
    Inicio_Lmom = paste(round(par_lmom$para, 6), collapse = ", ")
  )
}

ajustar_marginal <- function(x, distribucion) {

  x <- x[is.finite(x)]
  n <- length(x)

  if (n < 3) {
    stop("Muestra insuficiente para ajustar ", distribucion)
  }

  # ----------------------------------------------------------
  # 1. INICIALIZACIÓN MEDIANTE MOMENTOS-L
  # ----------------------------------------------------------
  lmr <- lmomco::lmoms(x)

  par_lmom <- tryCatch(
    lmomco::lmom2par(lmr, type = distribucion),
    error = function(e) NULL
  )

  if (is.null(par_lmom) || any(!is.finite(par_lmom$para))) {
    return(
      tibble(
        modelo = toupper(distribucion),
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = NA_integer_,
        Metodo = NA_character_,
        Estado = "No válido: no se obtuvo inicialización por momentos-L",
        Margen_soporte = NA_real_,
        Parametros = NA_character_,
        Inicio_Lmom = NA_character_
      )
    )
  }

  # Tratamiento específico y auditable de Weibull.
  if (distribucion == "wei") {
    return(ajustar_wei_mle(x, par_lmom))
  }

  par0 <- as.numeric(par_lmom$para)
  z0 <- a_escala_z(par0, distribucion, x)

  # ----------------------------------------------------------
  # 2. MÁXIMA VEROSIMILITUD
  # ----------------------------------------------------------
  inicios <- generar_multistart(z0)
  metodos <- c("BFGS", "Nelder-Mead")

  ajustes <- list()
  id <- 1

  for (inicio in inicios) {
    for (metodo in metodos) {

      fit <- tryCatch(
        optim(
          par = inicio,
          fn = nll_marginal,
          x = x,
          distribucion = distribucion,
          method = metodo,
          control = list(
            maxit = 20000,
            reltol = 1e-10
          )
        ),
        error = function(e) NULL
      )

      if (!is.null(fit) && is.finite(fit$value) && fit$value < 1e99) {
        ajustes[[id]] <- list(
          fit = fit,
          metodo = metodo
        )
        id <- id + 1
      }
    }
  }

  if (length(ajustes) == 0) {
    return(
      tibble(
        modelo = toupper(distribucion),
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = NA_integer_,
        Metodo = NA_character_,
        Estado = "No válido: no se obtuvo solución MLE",
        Margen_soporte = NA_real_,
        Parametros = NA_character_,
        Inicio_Lmom = paste(round(par0, 6), collapse = ", ")
      )
    )
  }

  convergentes <- ajustes[
    vapply(
      ajustes,
      function(a) identical(a$fit$convergence, 0L) ||
                  identical(a$fit$convergence, 0),
      logical(1)
    )
  ]

  candidatos <- if (length(convergentes) > 0) convergentes else ajustes

  valores <- vapply(
    candidatos,
    function(a) a$fit$value,
    numeric(1)
  )

  mejor <- candidatos[[which.min(valores)]]

  par_mle <- desde_escala_z(
    mejor$fit$par,
    distribucion,
    x
  )

  par_final <- tryCatch(
    lmomco::vec2par(par_mle, type = distribucion),
    error = function(e) NULL
  )

  if (is.null(par_final)) {
    return(
      tibble(
        modelo = toupper(distribucion),
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = mejor$fit$convergence,
        Metodo = mejor$metodo,
        Estado = "No válido: parámetros finales incompatibles",
        Margen_soporte = NA_real_,
        Parametros = NA_character_,
        Inicio_Lmom = paste(round(par0, 6), collapse = ", ")
      )
    )
  }

  densidad_final <- pdf_lmomco(
    x,
    distribucion,
    par_final
  )

  soporte_ok <- (
    length(densidad_final) == n &&
    all(is.finite(densidad_final)) &&
    all(densidad_final > 0)
  )

  if (!soporte_ok) {
    return(
      tibble(
        modelo = toupper(distribucion),
        n = n,
        logLik = NA_real_,
        AIC = Inf,
        BIC = Inf,
        Convergio = FALSE,
        Codigo_convergencia = mejor$fit$convergence,
        Metodo = mejor$metodo,
        Estado = "No válido: densidad o soporte incompatible",
        Margen_soporte = NA_real_,
        Parametros = paste(
        sprintf("%.17g", par_mle),
        collapse = ", "
        ),
        Inicio_Lmom = paste(round(par0, 6), collapse = ", ")
      )
    )
  }

  # ----------------------------------------------------------
  # 3. logLik MAXIMIZADA + AIC/BIC
  # ----------------------------------------------------------
  loglik <- sum(log(densidad_final))
  k <- length(par_mle)

  tibble(
    modelo = toupper(distribucion),
    n = n,
    logLik = loglik,
    AIC = -2 * loglik + 2 * k,
    BIC = -2 * loglik + log(n) * k,
    Convergio = mejor$fit$convergence == 0,
    Codigo_convergencia = mejor$fit$convergence,
    Metodo = mejor$metodo,
    Estado = ifelse(mejor$fit$convergence == 0, "Válido", "Revisar convergencia"),
    Margen_soporte = NA_real_,
    Parametros = paste(
    sprintf("%.17g", par_mle),
    collapse = ", "
    ),
    Inicio_Lmom = paste(round(par0, 6), collapse = ", ")
  )
}


In [ ]:
# ============================================================
# AJUSTE DE DISTRIBUCIONES CANDIDATAS POR CLUSTER
# ============================================================

# Se conservan EXACTAMENTE las familias candidatas de la tesis:
#
# Duración:
#   WEI, PE3, LN3, GAM
#
# Severidad:
#   PE3, GPA, LN3, GAM

ajustar_cluster <- function(datos, cluster) {

  duracion <- bind_rows(
    ajustar_marginal(datos$duracion, "wei"),
    ajustar_marginal(datos$duracion, "pe3"),
    ajustar_marginal(datos$duracion, "ln3"),
    ajustar_marginal(datos$duracion, "gam")
  ) %>%
    mutate(
      Grupo = paste("Cluster", cluster),
      Variable = "Duración (D)"
    )

  severidad <- bind_rows(
    ajustar_marginal(datos$severidad, "pe3"),
    ajustar_marginal(datos$severidad, "gpa"),
    ajustar_marginal(datos$severidad, "ln3"),
    ajustar_marginal(datos$severidad, "gam")
  ) %>%
    mutate(
      Grupo = paste("Cluster", cluster),
      Variable = "Severidad (S)"
    )

  bind_rows(duracion, severidad) %>%
  select(
    Grupo,
    Variable,
    Modelo = modelo,
    n,
    logLik,
    AIC,
    BIC,
    Convergio,
    Codigo_convergencia,
    Metodo,
    Estado,
    Margen_soporte,
    Parametros,
    Inicio_Lmom
  )
}

ajuste_marginales <- bind_rows(
  ajustar_cluster(c1, 1),
  ajustar_cluster(c2, 2),
  ajustar_cluster(c3, 3)
)

ajuste_marginales


In [ ]:
# ============================================================
# SELECCIÓN DE LAS MEJORES DISTRIBUCIONES
# AIC como criterio principal y contraste con BIC
# ============================================================

ajustes_validos <- ajuste_marginales %>%
  filter(
    Convergio,
    is.finite(AIC),
    is.finite(BIC),
    is.finite(logLik)
  )

mejores_aic <- ajustes_validos %>%
  group_by(Grupo, Variable) %>%
  slice_min(
    order_by = AIC,
    n = 1,
    with_ties = FALSE
  ) %>%
  ungroup()

mejores_bic <- ajustes_validos %>%
  group_by(Grupo, Variable) %>%
  slice_min(
    order_by = BIC,
    n = 1,
    with_ties = FALSE
  ) %>%
  ungroup() %>%
  select(
    Grupo,
    Variable,
    Modelo_BIC = Modelo,
    BIC_min = BIC
  )

mejores_marginales <- mejores_aic %>%
  left_join(
    mejores_bic,
    by = c("Grupo", "Variable")
  ) %>%
  mutate(
    Coincide_AIC_BIC = Modelo == Modelo_BIC
  ) %>%
  arrange(Grupo, Variable)

mejores_marginales

if (any(!mejores_marginales$Coincide_AIC_BIC)) {
  warning(
    "AIC y BIC no seleccionaron la misma distribución ",
    "en al menos una combinación cluster-variable. ",
    "Revise la tabla antes de fijar el modelo final."
  )
}


In [ ]:
# ============================================================
# TABLAS DEL AJUSTE MARGINAL PARA LA TESIS
# ============================================================

tabla_10 <- mejores_marginales %>%
  select(
    Grupo,
    Variable,
    Modelo,
    n,
    logLik,
    AIC,
    BIC,
    Parametros,
    Modelo_BIC,
    Coincide_AIC_BIC
  )

tabla_11 <- ajuste_marginales %>%
  filter(Variable == "Duración (D)") %>%
  arrange(Grupo, AIC)

tabla_12 <- ajuste_marginales %>%
  filter(Variable == "Severidad (S)") %>%
  arrange(Grupo, AIC)

tabla_10
tabla_11
tabla_12


In [ ]:
# ============================================================
# GUARDADO Y VERIFICACIÓN DE RESULTADOS
# ============================================================

archivos_salida <- c(
  ajuste = file.path(
    ruta_tablas,
    "ajuste_marginales_nspei3_por_cluster.csv"
  ),
  mejores = file.path(
    ruta_tablas,
    "mejores_marginales_nspei3_por_cluster.csv"
  ),
  tabla10 = file.path(
    ruta_tablas,
    "tabla_10_mejores_marginales_nspei3.csv"
  ),
  tabla11 = file.path(
    ruta_tablas,
    "tabla_11_marginales_duracion_nspei3.csv"
  ),
  tabla12 = file.path(
    ruta_tablas,
    "tabla_12_marginales_severidad_nspei3.csv"
  ),
  diagnostico = file.path(
    ruta_tablas,
    "diagnostico_convergencia_marginales_nspei3.csv"
  )
)

write.csv(
  ajuste_marginales,
  archivos_salida["ajuste"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  mejores_marginales,
  archivos_salida["mejores"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  tabla_10,
  archivos_salida["tabla10"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  tabla_11,
  archivos_salida["tabla11"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

write.csv(
  tabla_12,
  archivos_salida["tabla12"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

diagnostico_convergencia <- ajuste_marginales %>%
  select(
    Grupo,
    Variable,
    Modelo,
    n,
    logLik,
    AIC,
    BIC,
    Convergio,
    Codigo_convergencia,
    Metodo,
    Estado,
    Margen_soporte,
    Inicio_Lmom,
    Parametros
  )

write.csv(
  diagnostico_convergencia,
  archivos_salida["diagnostico"],
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

# Verificación visible en el cuaderno
cat("\n=== DISTRIBUCIONES SELECCIONADAS ===\n")
print(
  mejores_marginales %>%
    select(
      Grupo,
      Variable,
      Modelo,
      n,
      logLik,
      AIC,
      BIC,
      Modelo_BIC,
      Coincide_AIC_BIC
    )
)

cat("\n=== CONVERGENCIA DE TODOS LOS AJUSTES ===\n")
print(
  ajuste_marginales %>%
    select(
      Grupo,
      Variable,
      Modelo,
      Convergio,
      Codigo_convergencia,
      Metodo,
      Estado,
      Margen_soporte
    )
)

cat("\n=== ARCHIVOS GENERADOS ===\n")
print(
  tibble(
    archivo = basename(archivos_salida),
    existe = file.exists(archivos_salida),
    tamaño_bytes = file.info(archivos_salida)$size
  )
)
